In [13]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
import textwrap

In [4]:
county_path = Path("North_Carolina_State_and_County_Boundary_Polygons(1).geojson")
if not county_path.exists():
    county_path = Path("North_Carolina_State_and_County_Boundary_Polygons.geojson")
library_path = Path("Public_Libraries.geojson")

nc_counties = gpd.read_file(county_path)
libraries = gpd.read_file(library_path)

# Reproject to a common CRS
nc_counties = nc_counties.to_crs("EPSG:32119")
libraries = libraries.to_crs(nc_counties.crs)

print("Counties:", len(nc_counties))
print("Library locations:", len(libraries))

Counties: 100
Library locations: 382


In [6]:
joined_libraries = gpd.sjoin(
    libraries[["brnch_name", "geometry"]],
    nc_counties[["County", "geometry"]],
    how="inner",
    predicate="within"
)

# Count libraries per county, including counties with 0 mapped libraries.
counts = joined_libraries.groupby("County").size()
nc_counties["library_count"] = (
    nc_counties["County"].map(counts).fillna(0).astype(int)
)

print("Libraries matched to counties:", len(joined_libraries))
print(nc_counties[["County", "library_count"]].head())

# Create simplified geometries for mapping and locator maps
map_counties = nc_counties[["County", "geometry"]].copy()
map_counties["geometry"] = map_counties.geometry.simplify(
    tolerance=100, preserve_topology=True
)
locator_counties = nc_counties[["County", "geometry"]].copy()
locator_counties["geometry"] = locator_counties.geometry.simplify(
    tolerance=600, preserve_topology=True
)

Libraries matched to counties: 382
      County  library_count
0   Alamance              4
1  Alexander              1
2  Alleghany              1
3      Anson              1
4       Ashe              1


In [8]:
def county_map(county, ax):
    selected = map_counties[map_counties["County"] == county]
    points = joined_libraries[joined_libraries["County"] == county]

    selected.plot(ax=ax, facecolor="#e8f1e8", edgecolor="#275647", linewidth=1.5)
    if not points.empty:
        points.plot(ax=ax, color="#c04b35", markersize=35, edgecolor="white", linewidth=0.5)

    xmin, ymin, xmax, ymax = selected.total_bounds
    pad = max(xmax - xmin, ymax - ymin) * 0.07
    ax.set_xlim(xmin - pad, xmax + pad)
    ax.set_ylim(ymin - pad, ymax + pad)
    ax.set_title("Mapped public library locations", fontsize=12)
    ax.set_aspect("equal")
    ax.set_axis_off()


def locator_map(county, ax):
    locator_counties.plot(ax=ax, color="#e1e5e8", edgecolor="#ffffff", linewidth=0.35)
    selected = locator_counties[locator_counties["County"] == county]
    selected.plot(ax=ax, color="#2c7260", edgecolor="#214a40", linewidth=0.7)
    ax.set_title("Location in North Carolina", fontsize=10)
    ax.set_aspect("equal")
    ax.set_axis_off()


def county_info(county, ax):
    count = int(nc_counties.loc[nc_counties["County"] == county, "library_count"].iloc[0])

    #Somewhat confusing used AI
    if count == 1:
        sentence = "There is " + str(count) + " recorded public library in " + county + " County."
    else:
        sentence = "There are " + str(count) + " recorded public libraries in " + county + " County."

    ax.set_axis_off()
    ax.text(0.02, 0.96, "LIBRARY COUNT", fontsize=12, fontweight="bold",
            color="#275647", va="top", transform=ax.transAxes)
    ax.text(0.02, 0.75, str(count), fontsize=40, fontweight="bold",
            color="#152d2b", va="top", transform=ax.transAxes)
    ax.text(0.02, 0.43, textwrap.fill(sentence, width=29), fontsize=12,
            color="#25323a", va="top", transform=ax.transAxes, linespacing=1.5)


def county_figure(county, pdf):
    fig = plt.figure(figsize=(11, 8.5), facecolor="white")
    ax_county = fig.add_axes([0.05, 0.17, 0.63, 0.65])
    ax_locator = fig.add_axes([0.72, 0.51, 0.25, 0.27])
    ax_info = fig.add_axes([0.72, 0.17, 0.26, 0.30])

    county_map(county, ax_county)
    locator_map(county, ax_locator)
    county_info(county, ax_info)

    fig.text(0.05, 0.95, "GEOG 592  /  NORTH CAROLINA PUBLIC LIBRARIES ATLAS",
             fontsize=11, color="#2c7260", fontweight="bold")
    fig.text(0.05, 0.885, county + " County", fontsize=29, fontweight="bold", color="#152d2b")
    fig.text(0.05, 0.045,
             "Sources: uploaded NC county boundaries and Public Libraries GeoJSON | Counts from point-in-polygon spatial join",
             fontsize=8.5, color="#666666")
    pdf.savefig(fig, dpi=100)
    plt.close(fig)

In [14]:
with PdfPages("NC_Public_Libraries_Two_Sample_Pages.pdf") as pdf:
    for county in ["Durham", "Orange"]:
        county_figure(county, pdf)

print("Created NC_Public_Libraries_Two_Sample_Pages.pdf with 2 pages.")

Created NC_Public_Libraries_Two_Sample_Pages.pdf with 2 pages.
